# MAgent2 Mean-Field DSRQ

Train and evaluate MF-DSRQ on MAgent2 `adversarial_pursuit_v4`. The Bellman target uses the observed next EMA mean action and the TV-robust mean-field value operator.

In [1]:
from pathlib import Path
import sys

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "discrete_action_space").is_dir() and (path / "requirements.txt").exists():
            return path
    raise RuntimeError("Could not find the SRE-DQN repo root from the current notebook directory.")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from discrete_action_space.mean_field_dsrq.notebook_utils import (
    evaluate_mfdsrq_from_notebook,
    notebook_mfdsrq_config,
    train_mfdsrq_from_notebook,
)

cfg = notebook_mfdsrq_config(
    total_steps=20_000,
    num_envs=2,
    map_size=16,
    max_cycles=100,
    output_dir=str(ROOT / "discrete_action_space" / "mean_field_dsrq" / "runs" / "mean_field_dsrq_notebooks"),
    seed=42,
)
cfg

{'env_name': 'adversarial_pursuit_v4',
 'env_backend': 'magent2',
 'map_size': 16,
 'max_cycles': 100,
 'type_prefixes': {'predator': 'predator_', 'prey': 'prey_'},
 'n_nbr_actions_override': {'predator': 13, 'prey': 9},
 'epsilon_tv_start': 0.1,
 'epsilon_tv_end': 0.02,
 'epsilon_tv_decay_frac': 1.0,
 'beta_start': 1.0,
 'beta_end': 5.0,
 'beta_anneal_frac': 0.5,
 'epsilon_explore_start': 1.0,
 'epsilon_explore_end': 0.05,
 'epsilon_explore_decay_frac': 0.2,
 'gamma': 0.95,
 'lr': 0.0001,
 'batch_size': 64,
 'buffer_capacity': 200000,
 'learning_starts': 1000,
 'train_every': 4,
 'target_tau': 0.005,
 'grad_clip': 10.0,
 'ema_momentum': 0.5,
 'total_steps': 20000,
 'num_envs': 2,
 'seed': 42,
 'output_dir': '/home/wowthecoder/SRE-DQN/discrete_action_space/mean_field_dsrq/runs/mean_field_dsrq_notebooks',
 'log_interval': 1000,
 'save_interval': 20000,
 'use_gpu': True}

## Train

In [2]:
train_result = train_mfdsrq_from_notebook(cfg)
train_result

Device: cuda
Starting training: 20000 env steps, 2 envs
step=1,000  eps_tv=0.096  beta=1.40  eps_explore=0.763  grad_steps=584  episodes=10  sps=25  loss_predator=0.0082  ep_r_predator=-31.64  loss_prey=0.0108  ep_r_prey=-6.40
step=2,000  eps_tv=0.092  beta=1.80  eps_explore=0.525  grad_steps=1334  episodes=20  sps=25  loss_predator=0.0006  ep_r_predator=-31.20  loss_prey=0.0010  ep_r_prey=-5.40
step=3,000  eps_tv=0.088  beta=2.20  eps_explore=0.288  grad_steps=2084  episodes=30  sps=24  loss_predator=0.0007  ep_r_predator=-30.24  loss_prey=0.0048  ep_r_prey=-5.33
step=4,000  eps_tv=0.084  beta=2.60  eps_explore=0.050  grad_steps=2834  episodes=40  sps=25  loss_predator=0.0037  ep_r_predator=-28.99  loss_prey=0.0010  ep_r_prey=-5.60
step=5,000  eps_tv=0.080  beta=3.00  eps_explore=0.050  grad_steps=3584  episodes=50  sps=24  loss_predator=0.0007  ep_r_predator=-26.76  loss_prey=0.0003  ep_r_prey=-5.50
step=6,000  eps_tv=0.076  beta=3.40  eps_explore=0.050  grad_steps=4334  episodes=60 

{'run_dir': '/home/wowthecoder/SRE-DQN/discrete_action_space/mean_field_dsrq/runs/mean_field_dsrq_notebooks/adversarial_pursuit_v4/mf_dsrq_seed42',
 'total_steps': 20000,
 'completed_episodes': 200}

## Evaluate

In [3]:
eval_result = evaluate_mfdsrq_from_notebook(
    cfg,
    train_result["run_dir"],
    num_episodes=5,
    obs_noise_sigmas=[0.0],
)
eval_result

Loaded /home/wowthecoder/SRE-DQN/discrete_action_space/mean_field_dsrq/runs/mean_field_dsrq_notebooks/adversarial_pursuit_v4/mf_dsrq_seed42/ckpt_predator_final.pt
Loaded /home/wowthecoder/SRE-DQN/discrete_action_space/mean_field_dsrq/runs/mean_field_dsrq_notebooks/adversarial_pursuit_v4/mf_dsrq_seed42/ckpt_prey_final.pt

σ=0.00: predator=48.72±28.81  prey=-77.60±33.43


{'sigma=0.00': {'predator': {'mean_reward': 48.720000725984576,
   'std_reward': 28.81127599983378,
   'min_reward': 13.200000196695328,
   'max_reward': 91.2000013589859},
  'prey': {'mean_reward': -77.6,
   'std_reward': 33.434114314573975,
   'min_reward': -124.0,
   'max_reward': -36.0}}}

## Robustness Sweep

In [4]:
noise_sweep = evaluate_mfdsrq_from_notebook(
    cfg,
    train_result["run_dir"],
    num_episodes=5,
    obs_noise_sigmas=[0.0, 0.05, 0.10, 0.20],
)
noise_sweep

Loaded /home/wowthecoder/SRE-DQN/discrete_action_space/mean_field_dsrq/runs/mean_field_dsrq_notebooks/adversarial_pursuit_v4/mf_dsrq_seed42/ckpt_predator_final.pt
Loaded /home/wowthecoder/SRE-DQN/discrete_action_space/mean_field_dsrq/runs/mean_field_dsrq_notebooks/adversarial_pursuit_v4/mf_dsrq_seed42/ckpt_prey_final.pt

σ=0.00: predator=32.28±14.39  prey=-56.60±16.02

σ=0.05: predator=49.56±36.63  prey=-79.40±42.66

σ=0.10: predator=40.80±39.14  prey=-73.00±45.05

σ=0.20: predator=2.84±12.13  prey=-33.40±12.09


{'sigma=0.00': {'predator': {'mean_reward': 32.28000048100948,
   'std_reward': 14.394777045002376,
   'min_reward': 8.600000128149986,
   'max_reward': 52.60000078380108},
  'prey': {'mean_reward': -56.6,
   'std_reward': 16.01998751560063,
   'min_reward': -81.0,
   'max_reward': -31.0}},
 'sigma=0.05': {'predator': {'mean_reward': 49.56000073850155,
   'std_reward': 36.626854082498035,
   'min_reward': 20.60000030696392,
   'max_reward': 119.00000177323818},
  'prey': {'mean_reward': -79.4,
   'std_reward': 42.6595827452637,
   'min_reward': -161.0,
   'max_reward': -47.0}},
 'sigma=0.10': {'predator': {'mean_reward': 40.80000060796738,
   'std_reward': 39.140976554726315,
   'min_reward': 4.200000062584877,
   'max_reward': 115.400001719594},
  'prey': {'mean_reward': -73.0,
   'std_reward': 45.05108211796915,
   'min_reward': -158.0,
   'max_reward': -30.0}},
 'sigma=0.20': {'predator': {'mean_reward': 2.8400000423192977,
   'std_reward': 12.132205256531716,
   'min_reward': -14.2